In [1]:
from dotenv import load_dotenv
load_dotenv()
from src.data_loader import load_ground_truth
from src.store import get_documents_by_ids
from src.search import _build_patient_id_map, _patient_id_map
import ast

_build_patient_id_map()

df = load_ground_truth()
test_df = df[df['split'] == 'test'].reset_index(drop=True)
test_df = test_df[test_df['true_fhir_ids'].apply(lambda x: len(x) > 0)]

# Cuántos true IDs tiene cada pregunta en promedio
import numpy as np
n_true = test_df['true_fhir_ids'].apply(lambda x: sum(len(v) for v in x.values()))
print(f"Mean true IDs per question: {n_true.mean():.1f}")
print(f"Max true IDs: {n_true.max()}")
print(f"Min true IDs: {n_true.min()}")

Patient ID map built: 100 patients
Mean true IDs per question: 13.8
Max true IDs: 729
Min true IDs: 1


/Users/rnorel/.pyenv/versions/3.11.14/envs/CH/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/rnorel/Documents/Learning/Counsel/research-scientist-interview-main/src/search.py:12: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
from src.store import get_random_ids
import sqlite3

conn = sqlite3.connect("data/fhir_store.db")
cursor = conn.execute("""
    SELECT patient_uuid, COUNT(*) as n_resources 
    FROM (
        SELECT json_extract(doc, '$.subject.reference') as patient_uuid
        FROM resources
        WHERE json_extract(doc, '$.subject.reference') IS NOT NULL
    )
    GROUP BY patient_uuid
""")
import numpy as np
counts = [row[1] for row in cursor.fetchall()]
print(f"Mean: {np.mean(counts):.0f}")
print(f"Median: {np.median(counts):.0f}")
print(f"Min: {np.min(counts)}")
print(f"Max: {np.max(counts)}")
conn.close()

Mean: 9270
Median: 4257
Min: 677
Max: 52600
